# 1125 59일차

## 1. 모델 저장·불러오기

훈련시킨 모델을 파일로 남겨 두고, 다시 훈련하지 않고 꺼내 쓰는 것 (`m16_joblib1_save` ~ `m18_xgb2_load`)

훈련에는 시간이 드는데 예측만 하려고 매번 다시 훈련할 이유가 없음. 같은 모델로 평가나 제출을 반복할 때 저장분을 불러 씀

1. joblib : `joblib.dump(model, 경로)` / `joblib.load(경로)`
2. pickle : `pickle.dump(model, open(경로, 'wb'))` / `pickle.load(open(경로, 'rb'))`
3. XGBoost 자체 : `model.save_model(경로)` / 빈 모델을 만든 뒤 `model.load_model(경로)`

- **세 방식 모두 불러온 모델의 점수가 저장 전과 같음**
- `joblib`은 경로만 넘기고, `pickle`은 파일을 직접 열어 넘김. `'wb'`는 write binary, `'rb'`는 read binary
- XGBoost 자체 방식은 `XGBRegressor()`를 먼저 만들어 두고 그 객체에 불러옴
- 불러오는 쪽 코드에는 `# 3. 훈련`이 비어 있음. 불러오면 훈련 단계가 없음
- 데이터 분할과 스케일링은 불러오는 쪽에서도 같은 조건으로 다시 해야 함. 모델만 저장되고 스케일러는 저장되지 않음

#### cf) 저장 파일 확장자

- joblib·pickle은 확장자가 무엇으로 저장했는지 드러내는 이름일 뿐이고, 형식은 저장한 방식이 정함
- XGBoost 자체 방식은 반대로 확장자를 보고 형식을 고름. `.json`·`.ubj`가 아니면 불러올 때 형식을 추측했다는 경고가 뜸
- 경고는 `save_model`이 아니라 `load_model`에서 나오므로, 끄려면 불러오는 쪽 파일에서 `warnings.filterwarnings('ignore')`를 import 다음에 실행해야 함

## 2. 파라미터 묶어 넘기기

모델 파라미터를 딕셔너리에 모아 두고 한 번에 넣는 것 (`m16_joblib1_save` ~ `m18_xgb1_save`)

파라미터가 열 개를 넘으면 모델 호출 줄이 길어지고, 무엇을 넣었는지 한눈에 안 보임. 딕셔너리로 따로 두면 목록으로 읽히고 값만 고치면 됨

1. `parameters = {'n_estimators': 1000, 'learning_rate': 0.1, ...}` : 딕셔너리로 모아 둠
2. `XGBRegressor(**parameters)` : 모델을 만들 때 넣음
3. `model.set_params(**parameters)` : 이미 만든 모델에 넣음
4. `*` : 튜플을 위치 인자로 풀어 넘김
5. `**` : 딕셔너리를 키워드 인자로 풀어 넘김

- `**`는 `{'max_depth': 5}`를 `max_depth=5`로 바꿔 넣는 것
- 딕셔너리에 없는 파라미터는 `set_params`에 직접 적어 같이 넘길 수 있음
- 딕셔너리의 키 이름이 파라미터 이름과 같아야 함

### 2-1. 조기 종료

검증 점수가 나아지지 않으면 트리를 더 만들지 않고 멈추는 것

`n_estimators`를 크게 두면 그 수만큼 트리를 다 만드는데, 어느 지점부터는 점수가 더 좋아지지 않음. 멈출 지점을 직접 찾지 않고 모델이 판단하게 함

1. `early_stopping_rounds=20` : 20번 연속 나아지지 않으면 멈춤
2. `eval_set=[(x_test, y_test)]` : 멈출지 판단할 데이터
3. `verbose=1` : 라운드마다 평가 점수를 찍음
4. `model.best_iteration` : 점수가 가장 좋았던 지점

- **`n_estimators`는 상한이 되고, 실제로 만들어지는 트리 수는 그보다 적어짐**
- `best_iteration` 다음부터 조기 종료 라운드 수만큼 더 만들어 보고 멈추므로, 남는 트리는 가장 좋았던 지점보다 그만큼 많음
- 점수는 전체 트리가 아니라 `best_iteration`까지의 트리로 계산됨
- `eval_set`에 test를 주면 멈추는 지점을 test를 보고 정하게 됨. 그 점수를 성능으로 쓰면 test가 훈련에 쓰인 셈이 됨

## 3. 상관계수

두 컬럼이 같이 커지고 같이 작아지는 정도를 -1에서 1 사이로 나타낸 값 (`m19_correlation_coefficient1` ~ `m19_correlation_coefficient3`)

컬럼이 많을 때 어느 컬럼이 y와 관계가 있는지 미리 보면, 뺄 컬럼을 고르는 데 쓸 수 있음

1. `df.corr()` : 컬럼끼리의 상관계수를 표로 만듦
2. `1`에 가까우면 같이 커지고, `-1`에 가까우면 반대로 움직이고, `0`에 가까우면 직선 관계가 없음
3. 대각선은 자기 자신과의 값이라 항상 1
4. `sns.heatmap(data=df.corr(), square=True, annot=True, cbar=True)` : 표를 색으로 바꿈
   - `annot=True`는 칸에 숫자를 쓰고, `square=True`는 칸을 정사각형으로, `cbar=True`는 색 막대를 붙임

- **상관계수는 직선 관계만 봄. 곡선으로 관계가 있어도 0에 가깝게 나올 수 있음**
- 부호는 방향이고, 크기가 관계의 세기
- y와의 상관계수가 0에 가까운 컬럼은 혼자서는 y를 설명하지 못함
- seaborn은 따로 설치해야 함 (`pip install seaborn`)

![상관계수가 같은 값이어도 관계는 다를 수 있음](assets/correlation-linear-only.svg)

#### cf) 데이터프레임도 joblib·pickle로 저장

- 모델만이 아니라 데이터프레임도 같은 방식으로 저장하고 불러올 수 있음
- `joblib.dump(df, 경로)` / `joblib.load(경로)`
- 불러오는 쪽에서는 데이터를 다시 만들지 않고 바로 `df.corr()`을 씀
- `datasets.feature_names`를 컬럼 이름으로 주고 `df['Target'] = y`로 y를 붙이면, y와의 상관계수까지 한 표에서 봄